# 손목 PPG 품질 분류 (CPU)
AF 없이 저장된 RF로 좋은/나쁜 품질을 판정합니다. 기본 CPU 런타임으로 충분합니다.
실제 손목 성능은 독립 라벨 평가 전에는 알 수 없습니다. 원시 CSV에 장치 `sample_index`, `timestamp_s`가 필요합니다.

In [ ]:
from pathlib import Path
import subprocess, sys, json
REPO_REF = "codex/ppg-quality-standalone"  # 이 변경 브랜치; main 병합 후 main 사용 가능
repo = Path("/content/capdyssey_quality_only")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "https://github.com/dung-ho03/capdyssey.git", str(repo)], check=True)
print("실행 코드:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements-quality.txt")], check=True)
sys.path.insert(0, str(repo/"scripts"))
from quality_only import QualityOnly
MODEL = repo/"models/quality_af_v2_final"
model = QualityOnly(MODEL)
print("품질 모델 CPU 로드 완료 / 임계값:", model.threshold)

## 원시 CSV와 세션 목록 업로드
`sessions.csv` 필수 열: filename,subject_id,device_id,session_id,activity,channel,fs.
원시 파일은 sample_index,timestamp_s,선택한 채널 열을 포함해야 합니다.
이미 독립적으로 라벨링했다면 labels.csv도 함께 업로드하세요. 라벨 열: filename,window_index,quality(0/1 또는 빈칸).

In [ ]:
from google.colab import files
from datetime import datetime, timezone
DATA = Path("/content/quality_input_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
DATA.mkdir()
uploaded = files.upload()
for name, content in uploaded.items():
    if Path(name).name != name:
        raise ValueError("파일명에 경로를 포함하지 마세요")
    (DATA/name).write_bytes(content)
if not (DATA/"sessions.csv").is_file():
    raise ValueError("sessions.csv가 필요합니다. docs/QUALITY_ONLY.md 예시를 확인하세요.")

In [ ]:
from evaluate_quality_csv import run
OUT = Path("/content/quality_results_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
labels = DATA/"labels.csv"
report = run(DATA/"sessions.csv", DATA, MODEL, OUT, labels if labels.exists() else None)
print(json.dumps({k:v for k,v in report.items() if k not in ("files","by_group")}, indent=2, ensure_ascii=False))
print("저장:", OUT)
if not report["evaluation_complete"]:
    print("파일/라벨 오류가 있습니다. 부분 결과를 최종 평가로 사용하지 마세요.")

## 결과 다운로드
라벨이 없으면 성능 대신 예측·입력 검사 결과와 빈 label_template.csv가 생성됩니다.
모델 예측을 가린 상태로 원시 파형을 판독하고, 완성한 라벨을 labels.csv로 저장해 새 평가를 실행하세요.
입력 오류는 정답 bad가 아니며 평가 분모에서 따로 보고됩니다.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT), "zip", OUT)
files.download(archive)